# ✈️ Military Aircraft Detection - YOLOv8
Dataset: [MilitaryAircraftDetectionDataset](https://www.kaggle.com/datasets/a2015003713/militaryaircraftdetectiondataset) (~94 classes, CSV annotations)

## 1. Install

In [ ]:
!pip install ultralytics kaggle -q
from ultralytics import YOLO
import os, shutil, csv, random, yaml
from google.colab import drive, files
from PIL import Image
print('✅ Done')

## 2. Kaggle API

1. [Kaggle](https://www.kaggle.com) → Settings → API → Create New API Token
2. Çalıştır, `kaggle.json` seç

In [ ]:
uploaded = files.upload()
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
print('✅ Done')

## 3. Download Dataset

In [ ]:
!kaggle datasets download -d a2015003713/militaryaircraftdetectiondataset
!unzip -q militaryaircraftdetectiondataset.zip -d military_dataset/
print('✅ Downloaded')

## 4. Parse CSV → Discover All Classes

In [ ]:
DATASET_DIR = 'military_dataset/dataset'
all_annotations = []
class_set = set()

csv_files = sorted([f for f in os.listdir(DATASET_DIR) if f.endswith('.csv')])
print(f'Parsing {len(csv_files)} CSV files...')

for csv_file in csv_files:
    jpg_file = csv_file.replace('.csv', '.jpg')
    with open(os.path.join(DATASET_DIR, csv_file), 'r') as f:
        for row in csv.DictReader(f):
            class_set.add(row['class'])
            all_annotations.append((jpg_file, row['class'],
                int(row['xmin']), int(row['ymin']), int(row['xmax']), int(row['ymax']),
                int(row['width']), int(row['height'])))

class_names = sorted(class_set)
print(f'✅ {len(class_names)} classes, {len(all_annotations)} boxes')

## 5. Create YOLO Dataset (90/10 split)

In [ ]:
YOLO_DIR = 'yolo_dataset'
for s in ['train','val']:
    os.makedirs(os.path.join(YOLO_DIR, s, 'images'), exist_ok=True)
    os.makedirs(os.path.join(YOLO_DIR, s, 'labels'), exist_ok=True)

img_anns = {}
for a in all_annotations:
    img_anns.setdefault(a[0], []).append((class_names.index(a[1]), a[2], a[3], a[4], a[5], a[6], a[7]))

all_imgs = list(img_anns.keys())
random.seed(42); random.shuffle(all_imgs)
split = int(len(all_imgs)*0.9)

for split_name, img_list in [('train', all_imgs[:split]), ('val', all_imgs[split:])]:
    for jpg in img_list:
        shutil.copy(os.path.join(DATASET_DIR, jpg), os.path.join(YOLO_DIR, split_name, 'images', jpg))
        txt = []
        for cid, x1, y1, x2, y2, w, h in img_anns[jpg]:
            txt.append(f"{cid} {(x1+x2)/2/w:.6f} {(y1+y2)/2/h:.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}")
        with open(os.path.join(YOLO_DIR, split_name, 'labels', jpg.replace('.jpg','.txt')), 'w') as f:
            f.write('\n'.join(txt))

print(f'✅ Train: {len(all_imgs[:split])}, Val: {len(all_imgs[split:])}')

## 6. data.yaml

In [ ]:
with open(os.path.join(YOLO_DIR, 'data.yaml'), 'w') as f:
    yaml.dump({'train': os.path.abspath(os.path.join(YOLO_DIR,'train','images')),
               'val': os.path.abspath(os.path.join(YOLO_DIR,'val','images')),
               'nc': len(class_names), 'names': class_names}, f)
print(f'✅ {len(class_names)} classes')

## 7. Check GPU

In [ ]:
!nvidia-smi

## 8. TRAIN + AUTO-SAVE 🚀

**Eğitim bittiği anda Drive'a ve bilgisayara otomatik kaydeder.**

In [ ]:
# ═══ TRAIN ═══
model = YOLO('yolov8m.pt')
model.train(data=os.path.join(YOLO_DIR, 'data.yaml'), epochs=100, imgsz=640,
            batch=16, patience=15, device=0, workers=4,
            project='runs/train', name='military_aircraft', exist_ok=True)

# ═══ OTOMATİK KAYDET (eğitim bitince çalışır) ═══
BEST = 'runs/train/military_aircraft/weights/best.pt'

# Drive'a kaydet
drive.mount('/content/drive', force_remount=True)
!mkdir -p /content/drive/MyDrive/military_aircraft_model/
!cp {BEST} /content/drive/MyDrive/military_aircraft_model/
print('✅ Drive\'a kaydedildi!')

# Bilgisayara direkt indir
files.download(BEST)
print('✅ İndirme başladı!')

## 9. Evaluate

In [ ]:
m = model.val()
print(f"mAP@0.5: {m.box.map50:.3f}  mAP@0.5:0.95: {m.box.map:.3f}")

---
## ✅ Eğitim bitti!

### best.pt → app/models/ klasörüne koy → python3 gradio_app.py